# Test Lightweight Model on Challenge Data
This notebook loads the unseen "Challenge" applications (`.jsonl` files) from `ember2024_data`, processes their features, and predicts their maliciousness using our **Lightweight Top-100 Feature Model**.

In [13]:
import os
import sys
import glob
import json
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Ensure the paths to the virtual environment(s) are added BEFORE importing packages
base_dir = os.getcwd()
for venv_folder in ["venv", ".venv"]:
    site_packages = os.path.join(base_dir, venv_folder, "Lib", "site-packages")
    if os.path.exists(site_packages) and site_packages not in sys.path:
        sys.path.insert(0, site_packages)
        break

global_venv_site_packages = r"Z:\ai project\.venv\Lib\site-packages"
if os.path.exists(global_venv_site_packages) and global_venv_site_packages not in sys.path:
    sys.path.insert(0, global_venv_site_packages)

# Import the feature extractor to convert the raw JSON to mathematical feature vectors
try:
    from thrember.features import PEFeatureExtractor
    print("Features extractor imported successfully.")
    extractor = PEFeatureExtractor()
except ImportError:
    print("Error: 'thrember' not found. We cannot convert raw JSON files without the extractor.")
    extractor = None

Features extractor imported successfully.


In [14]:
# 1. Setup Paths & Load the Lightweight Model
DATASET_DIR = r"C:\Users\him\ember2024_project\ember2024_data"
MODEL_PATH = os.path.join(r"Z:\ember2024_train_data", "ember_model_reduced_100.txt")
INDICES_PATH = os.path.join(r"Z:\ember2024_train_data", "ember_reduced_100_indices.npy")

print(f"Loading Lightweight model from {MODEL_PATH}")
model = lgb.Booster(model_file=MODEL_PATH)

print(f"Loading Top 100 Indices...")
top_100_indices = np.load(INDICES_PATH)

Loading Lightweight model from Z:\ember2024_train_data\ember_model_reduced_100.txt
Loading Top 100 Indices...


In [15]:
# 2. Get the list of Challenge files
jsonl_files = glob.glob(os.path.join(DATASET_DIR, "*_challenge_malicious.jsonl"))
print(f"Found {len(jsonl_files)} Challenge JSONL files.")

# To avoid using too much RAM, we will process one file as an example.
# You can change this loop to process all of them later if you want.
demo_files = jsonl_files[:2] 

all_predictions = []
all_labels = []

print("\n🚀 Beginning Challenge Prediction...")
for file_path in demo_files:
    print(f"\nProcessing: {os.path.basename(file_path)}")
    
    # We know their labels are Malicious (1) because of the filename
    y_true = 1 
    
    X_batch_full = []
    
    with open(file_path, "r") as f:
        for line in f:
            try:
                raw_json = json.loads(line.strip())
                # EMBER's Vectorize function takes the raw dictionary and outputs the (2381,) array
                vec = extractor.process_raw_features(raw_json)
                X_batch_full.append(vec)
            except Exception as e:
                continue
                
    X_batch_full = np.array(X_batch_full)
    
    if len(X_batch_full) == 0:
        continue
        
    print(f"Extracted {len(X_batch_full)} app records.")
    
    # 3. Slice to ONLY the Top 100 Features!
    X_batch_reduced = X_batch_full[:, top_100_indices]
    
    # 4. Predict
    preds_prob = model.predict(X_batch_reduced)
    preds_binary = (preds_prob > 0.5).astype(int)
    
    # Track metrics
    all_predictions.extend(preds_binary)
    all_labels.extend([y_true] * len(preds_binary))
    
    acc = np.mean(preds_binary == y_true)
    print(f"File Accuracy (Catch Rate): {acc * 100:.2f}%")

print("\n" + "="*50)
final_acc = accuracy_score(all_labels, all_predictions)
print(f"🏁 Final Overall Catch Rate on selected Challenge apps: {final_acc * 100:.2f}%")
print("="*50)

Found 64 Challenge JSONL files.

🚀 Beginning Challenge Prediction...

Processing: 2023-09-24_2023-09-30_challenge_malicious.jsonl
Extracted 44 app records.
File Accuracy (Catch Rate): 86.36%

Processing: 2023-10-01_2023-10-07_challenge_malicious.jsonl
Extracted 59 app records.
File Accuracy (Catch Rate): 76.27%

🏁 Final Overall Catch Rate on selected Challenge apps: 80.58%


### 5. Scientific Comparison: The "Full Model" (2381 Features)
Let's test our theory. We will load the massive original model and test it against the exact same two Challenge files. 
If the Full Model catches >90%, it proves the missing 2,281 features contained necessary details for edge-case zero-days. If the Full Model also gets around 80%, it proves this malware is fundamentally designed to evade Static Analysis, regardless of feature count!

In [16]:
# 1. Load the original massive model
FULL_MODEL_PATH = os.path.join(r"Z:\ember2024_train_data", "ember_model_tuned_full.txt")
print(f"Loading Full Mamba Model from {FULL_MODEL_PATH}")
full_model = lgb.Booster(model_file=FULL_MODEL_PATH)

all_predictions_full = []
all_labels_full = []

print("\n🚀 Beginning Full Model Prediction (Using ALL 2381 Features)...")
for file_path in demo_files:
    print(f"\nProcessing: {os.path.basename(file_path)}")
    y_true = 1 
    
    X_batch_full = []
    
    with open(file_path, "r") as f:
        for line in f:
            try:
                raw_json = json.loads(line.strip())
                vec = extractor.process_raw_features(raw_json)
                X_batch_full.append(vec)
            except Exception as e:
                continue
                
    X_batch_full = np.array(X_batch_full)
    
    if len(X_batch_full) == 0:
        continue
        
    print(f"Extracted {len(X_batch_full)} app records.")
    
    # Notice we DO NOT slice X_batch_full here. We give it all 2381 columns.
    preds_prob_full = full_model.predict(X_batch_full)
    preds_binary_full = (preds_prob_full > 0.5).astype(int)
    
    all_predictions_full.extend(preds_binary_full)
    all_labels_full.extend([y_true] * len(preds_binary_full))
    
    acc = np.mean(preds_binary_full == y_true)
    print(f"Full Model Accuracy (Catch Rate): {acc * 100:.2f}%")

print("\n" + "="*50)
final_acc_full = accuracy_score(all_labels_full, all_predictions_full)
print(f"🔬 Lightweight Top-100 Catch Rate : {final_acc * 100:.2f}%")
print(f"🔬 Massive Full-2381 Catch Rate   : {final_acc_full * 100:.2f}%")
print("="*50)

Loading Full Mamba Model from Z:\ember2024_train_data\ember_model_tuned_full.txt

🚀 Beginning Full Model Prediction (Using ALL 2381 Features)...

Processing: 2023-09-24_2023-09-30_challenge_malicious.jsonl
Extracted 44 app records.
Full Model Accuracy (Catch Rate): 63.64%

Processing: 2023-10-01_2023-10-07_challenge_malicious.jsonl
Extracted 59 app records.
Full Model Accuracy (Catch Rate): 54.24%

🔬 Lightweight Top-100 Catch Rate : 80.58%
🔬 Massive Full-2381 Catch Rate   : 58.25%
